All the experiments in this notebook are controlled by a **single setting**: the factual event indicator `E_FACTUAL` (0 or 1) in the config cell below.

- Set `E_FACTUAL = 1` to treat the **event** series as factual and the **no-event** series as counterfactual (default).
- Set `E_FACTUAL = 0` to treat the **no-event** series as factual and the **event** series as counterfactual.




In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.optimizers import Adam

from models import CEPAE, CVAE, CAAE
from models.baselines import ForecastModel, EventPredictor
from models.caae_trainer import train as train_caae
from models.adversarial_forecast_model import AdversarialForecastModel, Trainer

from data.semisynthetic_dataset import prepare_rossmann_datasets

from metrics.cf_metrics import (
    counterfactual_mae_mbe,
    axiomatic_metrics,
    added_variations_relative,
)

In [ ]:
pwd

In [ ]:
# -----------------------------
# Experiment configuration
# -----------------------------
# Path to Rossmann "train.csv" (download from Kaggle and place it here)
CSV_PATH = "data/rossmann/train.csv"

# Window sizes (must match the model definitions used below)
LOOKBACK_DAYS = 28
HORIZON_DAYS = 21

# Reproducibility
SEED = 100

# Factual / counterfactual event indicator (0 = no-event, 1 = event)
E_FACTUAL = 1
E_COUNTERFACTUAL = 1 - E_FACTUAL  # keep opposite by default

# Semi-synthetic intervention used to create the "event" outcome sequence from the baseline one
EVENT_SCALE = (1.1, 1.2, 1.3)  # (multiplier at t=0, t=1, t>=2)


# Data Preprocessing

In [ ]:
# Build datasets (train/eval/test) + select factual/counterfactual views
data = prepare_rossmann_datasets(
    csv_path=CSV_PATH,
    lookback=LOOKBACK_DAYS,
    horizon=HORIZON_DAYS,
    seed=SEED,
    e_factual=E_FACTUAL,
)

# Train split
x_train = data["train"]["x"]
y_train = data["train"]["y"]
train_labels = data["train"]["labels"]

# Eval split (paired outcome versions: y_data_0_* = no-event, y_data_1_* = event)
x_data_eval = data["eval"]["x"]
y_data_0_eval = data["eval"]["y0"]
y_data_1_eval = data["eval"]["y1"]

y_factual_eval = data["eval"]["y_factual"]
y_counterfactual_eval = data["eval"]["y_counterfactual"]

label_real = data["eval"]["label_factual"]
label_cf = data["eval"]["label_counterfactual"]

# Test split
x_data_test = data["test"]["x"]
y_data_0_test = data["test"]["y0"]
y_data_1_test = data["test"]["y1"]

y_factual_test = data["test"]["y_factual"]
y_counterfactual_test = data["test"]["y_counterfactual"]

label_real_test = data["test"]["label_factual"]
label_cf_test = data["test"]["label_counterfactual"]

# Backwards-compatible aliases used later in the notebook
eval_labels = label_real
test_labels = label_real_test
label_real_eval = label_real
label_cf_eval = label_cf


# Train predictor for effectiveness metric

In [ ]:
predictor = EventPredictor()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
predictor.compile(optimizer=optimizer, loss=tf.keras.losses.BinaryCrossentropy(), metrics = ["accuracy"])
predictor.fit(y_train, train_labels, epochs= 200, batch_size=32, verbose=0)


# CEPAE

In [ ]:
latent_dim = 8
Lambda = 0.09
batch_size = 32
series_size=21
model_cepae = CEPAE(seq_len = series_size, latent_dim = latent_dim, feat_dim = 1, hidden_layer_sizes = [100,200], Lambda=Lambda)
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
model_cepae.compile(optimizer, loss=model_cepae.loss_, metrics=[model_cepae.reconstruction, model_cepae.regularization])
history = model_cepae.fit(x =[train_labels, x_train, y_train], y=y_train, validation_data = [[eval_labels, x_data_eval, y_factual_eval], y_factual_eval], epochs=350, batch_size=batch_size, verbose=0)

In [ ]:
# Counterfactual prediction quality (MAE / MBE)
pred = np.asarray(model_cepae.cf_generation(label_real=label_real_test, label_cf=label_cf_test, x=x_data_test, y=y_factual_test))
m = counterfactual_mae_mbe(y_counterfactual_test, pred)
print(f"MAE: {m['cf_mae']:.6f}  MBE: {m['cf_mbe']:.6f}")


In [ ]:
# Added variations metrics (Total / Altered / Unaltered)
seq_length = 21
cf_from_y = lambda y: np.asarray(
    model_cepae.cf_generation(label_real=label_real_test, label_cf=label_cf_test, x=x_data_test, y=y)
)
m = added_variations_relative(cf_from_y, y_factual=y_factual_test, seq_length=seq_length, ini_start=2, num_windows=3, window_len=4)
print(f"Total difference: {m['total_rel']:.6f}  Step difference: {m['altered_steps_rel']:.6f}  Unaltered difference: {m['unaltered_steps_rel']:.6f}")


In [ ]:
# Composition / Reversibility / Effectiveness (axiomatic metrics)
m = axiomatic_metrics(
    model_cepae, predictor,
    label_real=label_real_test, label_cf=label_cf_test,
    x=x_data_test, y_factual=y_factual_test,
)
print(f"Composition: {m['composition']:.6f}  Reversibility: {m['reversibility']:.6f}  Effectiveness: {m['effectiveness']:.6f}")


# CVAE

In [ ]:
latent_dim = 6
recon_weight = 200
batch_size = 32
series_size=21
vae_model = CVAE(seq_len = series_size, latent_dim = latent_dim, feat_dim = 1, hidden_layer_sizes = [100,200], recon_weight=recon_weight)
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
vae_model.compile(optimizer, loss=vae_model.loss_, metrics=[vae_model.reconstruction, vae_model.kl])
history = vae_model.fit(x =[train_labels, x_train, y_train], y=y_train, validation_data = [[eval_labels, x_data_eval, y_factual_eval], y_factual_eval], epochs=250, batch_size=batch_size, verbose=0)


In [ ]:
# Counterfactual prediction quality (MAE / MBE)
pred = np.asarray(vae_model.cf_generation(label_real=label_real_test, label_cf=label_cf_test, x=x_data_test, y=y_factual_test))
m = counterfactual_mae_mbe(y_counterfactual_test, pred)
print(f"MAE: {m['cf_mae']:.6f}  MBE: {m['cf_mbe']:.6f}")


In [ ]:
# Added variations metrics (Total / Altered / Unaltered)
seq_length = 21
cf_from_y = lambda y: np.asarray(
    vae_model.cf_generation(label_real=label_real_test, label_cf=label_cf_test, x=x_data_test, y=y)
)
m = added_variations_relative(cf_from_y, y_factual=y_factual_test, seq_length=seq_length, ini_start=2, num_windows=3, window_len=4)
print(f"Total difference: {m['total_rel']:.6f}  Step difference: {m['altered_steps_rel']:.6f}  Unaltered difference: {m['unaltered_steps_rel']:.6f}")


In [ ]:
# Composition / Reversibility / Effectiveness (axiomatic metrics)
m = axiomatic_metrics(
    vae_model, predictor,
    label_real=label_real_test, label_cf=label_cf_test,
    x=x_data_test, y_factual=y_factual_test,
)
print(f"Composition: {m['composition']:.6f}  Reversibility: {m['reversibility']:.6f}  Effectiveness: {m['effectiveness']:.6f}")


# CAAE

In [ ]:
# data preprocessing for CAAE
train_labels, x_train, y_train, label_real_test, x_data_test, y_factual_test = train_labels.astype("float32"), x_train.astype("float32"), y_train.astype("float32"), label_real_test.astype("float32"), x_data_test.astype("float32"), y_factual_test.astype("float32")

train_dataset = tf.data.Dataset.from_tensor_slices((train_labels, x_train, y_train))
train_dataset = train_dataset.shuffle(buffer_size=2024).batch(32)

test_dataset = tf.data.Dataset.from_tensor_slices((label_real_test, x_data_test, y_factual_test))
test_dataset = test_dataset.shuffle(buffer_size=2024).batch(32)

In [ ]:
latent_dim = 7
series_size = 21
batch_size = 32
model_caae = CAAE(seq_len = series_size, latent_dim = latent_dim, feat_dim = 1, hidden_layer_sizes = [100,200])
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
max_iterations = 500000
train_caae(model_caae, train_dataset, test_dataset, epochs=350, batch_size=32, optimizer=optimizer, max_iterations=max_iterations, max_lambda=8.9)

In [ ]:
# Counterfactual prediction quality (MAE / MBE)
pred = np.asarray(model_caae.cf_generation(label_real=label_real_test, label_cf=label_cf_test, x=x_data_test, y=y_factual_test))
m = counterfactual_mae_mbe(y_counterfactual_test, pred)
print(f"MAE: {m['cf_mae']:.6f}  MBE: {m['cf_mbe']:.6f}")


In [ ]:
# Added variations metrics (Total / Altered / Unaltered)
seq_length = 21
cf_from_y = lambda y: np.asarray(
    model_caae.cf_generation(label_real=label_real_test, label_cf=label_cf_test, x=x_data_test, y=y)
)
m = added_variations_relative(cf_from_y, y_factual=y_factual_test, seq_length=seq_length, ini_start=2, num_windows=3, window_len=4)
print(f"Total difference: {m['total_rel']:.6f}  Step difference: {m['altered_steps_rel']:.6f}  Unaltered difference: {m['unaltered_steps_rel']:.6f}")


In [ ]:
# Composition / Reversibility / Effectiveness (axiomatic metrics)
m = axiomatic_metrics(
    model_caae, predictor,
    label_real=label_real_test, label_cf=label_cf_test,
    x=x_data_test, y_factual=y_factual_test,
)
print(f"Composition: {m['composition']:.6f}  Reversibility: {m['reversibility']:.6f}  Effectiveness: {m['effectiveness']:.6f}")


# Forecast

In [ ]:
# Baseline: conditional LSTM forecaster (predict Y given history X and event indicator E)
pred_steps = HORIZON_DAYS

# forecast_model outputs shape (B, pred_steps), so we squeeze targets to (B, pred_steps)
y_train_flat = np.squeeze(y_train)
y_factual_eval_flat = np.squeeze(y_factual_eval)

model_forecast = ForecastModel(pred_steps=pred_steps)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model_forecast.compile(optimizer=optimizer, loss="mse", metrics=["mae"])

model_forecast.fit(
    [train_labels, x_train],
    y_train_flat,
    epochs=500,
    batch_size=32,
    verbose=0,
    validation_data=([label_real, x_data_eval], y_factual_eval_flat),
)


In [ ]:
# Counterfactual prediction quality (MAE / MBE)
pred = np.asarray(model_forecast([label_cf_test, x_data_test]))
m = counterfactual_mae_mbe(y_counterfactual_test, pred)
print(f"MAE: {m['cf_mae']:.6f}  MBE: {m['cf_mbe']:.6f}")


# Adversarially Balanced Forecast

In [ ]:
# Baseline: AdversarialForecastModel (AB-LSTM)
# This model outputs shape (B, pred_steps), so we use flattened targets (B, pred_steps).
y_train_flat = np.squeeze(y_train)
y_factual_eval_flat = np.squeeze(y_factual_eval)

train_labels_f32 = train_labels.astype("float32")
x_train_f32 = x_train.astype("float32")
y_train_f32 = y_train_flat.astype("float32")

eval_labels_f32 = label_real.astype("float32")
x_eval_f32 = x_data_eval.astype("float32")
y_eval_f32 = y_factual_eval_flat.astype("float32")

train_dataset = (
    tf.data.Dataset.from_tensor_slices((train_labels_f32, x_train_f32, y_train_f32))
    .shuffle(2048)
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)

eval_dataset = (
    tf.data.Dataset.from_tensor_slices((eval_labels_f32, x_eval_f32, y_eval_f32))
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)


In [ ]:
# (datasets already created in the previous cell)


In [ ]:
# 1) infer how many steps we predict
pred_steps = y_train_flat.shape[1]          # 10 in your example

# 2)  model & optimiser
model     = AdversarialForecastModel(pred_steps)
optimizer = Adam(1e-3)

# 3)  trainer –  we need max_steps for the λ schedule
epochs           = 200                  # tweak as you like
steps_per_epoch  = tf.data.experimental.cardinality(train_dataset).numpy()
max_steps        = 40 * steps_per_epoch   

trainer = Trainer(model,
                  optimizer,
                  max_steps=max_steps,
                  max_lambda=1.0)       # or 2.0 etc.

In [ ]:
trainer.fit(train_dataset,
            eval_dataset,
            epochs=epochs)

In [ ]:
# Counterfactual prediction quality (MAE / MBE) for AB-LSTM
pred = np.asarray(model((label_cf_test.astype("float32"), x_data_test.astype("float32"))))
m = counterfactual_mae_mbe(y_counterfactual_test, pred)
print(f"MAE: {m['cf_mae']:.6f}  MBE: {m['cf_mbe']:.6f}")
